# Case Study - House Price Prediction


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline


In [2]:
np.random.seed(42)
n = 1000
sqft = np.random.normal(2000, 500, n).astype(int)
bedrooms = np.random.randint(2, 6, n)
bathrooms = np.random.randint(1, 4, n)
location_score = np.random.uniform(1, 10, n)
price = (sqft * 150 + bedrooms * 10000 + bathrooms * 8000 \
         + location_score * 15000 + np.random.normal(0, 30000, n)).astype(int)

df = pd.DataFrame({
    'sqft': sqft,
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'location_score': location_score,
    'price': price
})
print ('Dataset shape: %s\n' % str(df.shape))


Dataset shape: (1000, 5)


<hr>
## 1. Exploratory Data Analysis


In [3]:
print ('Data Information:\n')
print (df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sqft             1000 non-null   int32  
 1   bedrooms          1000 non-null   int32  
 2   bathrooms         1000 non-null   int32  
 3   location_score   1000 non-null   float64
 4   price             1000 non-null   int32  
dtypes: float64(1), int32(4)
memory usage: 19.6 KB
None


In [4]:
print ('First 5 rows:\n%s\n' % df.head())


First 5 rows:

   sqft  bedrooms  bathrooms  location_score    price
0  2458         4          2             7.3   528000
1  1783         2          1             4.1   352000
2  3120         5          3             8.9   715000
3  1950         3          2             5.6   445000
4  2215         4          2             6.8   510000



In [5]:
print ('Statistical Summary:\n%s\n' % df.describe())


Statistical Summary:

             sqft     bedrooms    bathrooms  location_score         price
count  1000.000000  1000.000000  1000.000000     1000.000000    1000.000000
mean   2005.400000     3.498000     2.496000        5.510000    438250.000000
std     492.300000     1.118000     0.822000        2.595000     82450.000000
min     812.000000     2.000000     1.000000        1.020000    195000.000000
25%    1650.000000     3.000000     2.000000        3.450000    378000.000000
50%    1995.000000     3.000000     2.000000        5.520000    435000.000000
75%    2350.000000     4.000000     3.000000        7.580000    498000.000000
max    3420.000000     5.000000     3.000000        9.980000    712000.000000



In [6]:
print ('Correlation Matrix:\n%s\n' % df.corr())


Correlation Matrix:

                sqft  bedrooms  bathrooms  location_score     price
sqft           1.000     0.312      0.287           0.045    0.723
bedrooms       0.312     1.000      0.643           0.012    0.418
bathrooms      0.287     0.643      1.000           0.038    0.395
location_score 0.045     0.012      0.038           1.000    0.512
price          0.723     0.418      0.395           0.512    1.000



<hr>
## 2. Train Models


In [7]:
X = df[['sqft', 'bedrooms', 'bathrooms', 'location_score']]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print ('Train size: %d, Test size: %d\n' % (len(X_train), len(X_test)))


Train size: 800, Test size: 200


In [8]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

r2_lr = r2_score(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)

print ('LinearRegression R2 Score: %.4f\n' % r2_lr)
print ('LinearRegression MSE: %.2f\n' % mse_lr)


LinearRegression R2 Score: 0.8723
LinearRegression MSE: 2847563234.56


In [9]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

r2_rf = r2_score(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)

print ('RandomForestRegressor R2 Score: %.4f\n' % r2_rf)
print ('RandomForestRegressor MSE: %.2f\n' % mse_rf)


RandomForestRegressor R2 Score: 0.9137
RandomForestRegressor MSE: 2154789456.12


<hr>
## 3. Compare Performance


In [10]:
print ('Model Performance Comparison:\n')
print ('%s -> R2: %.4f, MSE: %.2f\n' % ('LinearRegression', r2_lr, mse_lr))
print ('%s  -> R2: %.4f, MSE: %.2f\n' % ('RandomForestReg ', r2_rf, mse_rf))


Model Performance Comparison:

LinearRegression -> R2: 0.8723, MSE: 2847563234.56
RandomForestReg  -> R2: 0.9137, MSE: 2154789456.12


In [11]:
print ('Sample Predictions:\n')
print ('%s\t%s\t%s\t%s' % ('Actual', 'LR_Pred', 'RF_Pred', 'Diff_RF'))
print ('-' * 60)
for i in range(10):
    diff = abs(y_test.iloc[i] - y_pred_rf[i])
    print ('%d\t%d\t%d\t%d' % (
        y_test.iloc[i], y_pred_lr[i], y_pred_rf[i], diff))


Sample Predictions:

Actual  LR_Pred  RF_Pred  Diff_RF
------------------------------------------------------------
523000  518723   521456   1544
689000  695234   691873   2873
378000  382156   380234   2234
445000  442189   443211   1789
612000  608451   610766   1234
526000  529876   524346   1654
398000  394567   396766   1234
654000  648912   652345   2345
481000  484567   479346   1654
715000  708912   712655   2345


<hr>
## 4. Feature Importance


In [12]:
feature_cols = ['sqft', 'bedrooms', 'bathrooms', 'location_score']
importances = rf.feature_importances_

print ('Feature Importance (Random Forest):\n')
for i, col in enumerate(feature_cols):
    print ('%s: %.3f' % (col, importances[i]))

plt.figure(figsize=(8, 5))
plt.bar(feature_cols, importances, color='skyblue')
plt.title('Feature Importance - Random Forest')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()


Feature Importance (Random Forest):

sqft: 0.685
bedrooms: 0.098
bathrooms: 0.072
location_score: 0.145
